#  Non-Gunshot Duration Balancer

**Problem**: The current dataset has a **duration bias** — all non-gunshot clips are ~5s while gunshot clips vary from 93ms–13s. The MFCC features encode this difference via silence padding, causing the model to learn "short = gunshot" instead of acoustic features.

**Solution**: Randomly crop each non-gunshot file to a duration sampled from the actual gunshot duration distribution, then re-extract MFCCs.

### Pipeline Steps
1. Load gunshot durations from `trimmed_gunshots/`
2. Randomly crop each non-gunshot file to a sampled gunshot duration
3. Save cropped files to `trimmed_nongunshots/`
4. Re-extract MFCC features from both trimmed sets
5. Generate `dataset_features_balanced.csv`

In [ ]:
import warnings
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings('ignore')

def resolve_data_root():
    current = Path.cwd().resolve()
    expected_subdirs = ('edge-collected-gunshot-audio', 'gun', 'audio', 'sound')
    best_data_dir = None
    best_score = -1
    best_depth = 10**9

    for candidate in [current, *current.parents]:
        data_dir = candidate / 'Data'
        if not data_dir.exists():
            continue
        score = sum((data_dir / subdir).exists() for subdir in expected_subdirs)
        depth = len(data_dir.parts)
        if score > best_score or (score == best_score and depth < best_depth):
            best_data_dir = data_dir
            best_score = score
            best_depth = depth

    if best_data_dir is None:
        raise FileNotFoundError("Could not locate a valid Data folder.")

    return best_data_dir, best_score

# ──── CONFIG ────
BASE_DIR, DATA_MATCH_SCORE = resolve_data_root()
PROJECT_ROOT = BASE_DIR.parent
OUTPUT_DIR = BASE_DIR / 'Output'
TRIMMED_GUNSHOTS_DIR = OUTPUT_DIR / 'trimmed_gunshots'
TRIMMED_NONGUNSHOTS_DIR = OUTPUT_DIR / 'trimmed_nongunshots'
CSV_PATH = OUTPUT_DIR / 'dataset_features_balanced.csv'

# Class 0 source directories (non-gunshot)
CLASS_0_DIRS = [BASE_DIR / 'audio', BASE_DIR / 'sound']

# Audio / MFCC settings (MUST match audio_pipeline.py exactly)
SR = 22050
N_MFCC = 40
MAX_LEN = SR * 4  # 4 seconds

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Config loaded.')
print(f'  Project root         : {PROJECT_ROOT}')
print(f'  Data root            : {BASE_DIR} (match score: {DATA_MATCH_SCORE}/4)')
print(f'  Trimmed gunshots dir : {TRIMMED_GUNSHOTS_DIR}')
print(f'  Output dir           : {TRIMMED_NONGUNSHOTS_DIR}')
print(f'  Balanced CSV         : {CSV_PATH}')
print('  Label mapping        : class 1 -> gun + edge-collected, class 0 -> audio + sound')

## Step 1 — Load Gunshot Duration Distribution

In [ ]:
# Collect all trimmed gunshot files and measure their durations
gunshot_files = sorted([f for f in TRIMMED_GUNSHOTS_DIR.rglob('*.wav') if '__MACOSX' not in str(f) and not f.name.startswith('._')])
print(f'Found {len(gunshot_files)} trimmed gunshot files.')

gunshot_durations_samples = []
for f in tqdm(gunshot_files, desc='Measuring gunshot durations'):
    try:
        y, sr = librosa.load(str(f), sr=SR)
        gunshot_durations_samples.append(len(y))
    except Exception as e:
        print(f'  [SKIP] {f.name}: {e}')

gunshot_durations_ms = [round((s / SR) * 1000, 2) for s in gunshot_durations_samples]

print(f'\nGunshot Duration Stats (ms):')
print(f'  Min    : {min(gunshot_durations_ms):.0f} ms')
print(f'  Max    : {max(gunshot_durations_ms):.0f} ms')
print(f'  Mean   : {np.mean(gunshot_durations_ms):.0f} ms')
print(f'  Median : {np.median(gunshot_durations_ms):.0f} ms')
print(f'  Std    : {np.std(gunshot_durations_ms):.0f} ms')


In [ ]:
# Visualize the gunshot duration distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(gunshot_durations_ms, bins=60, color='#e74c3c', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('Duration (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Gunshot Duration Distribution')
axes[0].axvline(np.median(gunshot_durations_ms), color='yellow', linestyle='--', label=f'Median: {np.median(gunshot_durations_ms):.0f}ms')
axes[0].legend()

# Compare with class 0 (all 5000ms)
axes[1].hist(gunshot_durations_ms, bins=60, color='#e74c3c', alpha=0.7, label='Gunshot (class 1)', edgecolor='black')
axes[1].axvline(5000, color='#3498db', linewidth=3, linestyle='--', label='Non-gunshot: 5000ms (all)')
axes[1].set_xlabel('Duration (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Duration Bias: Before Balancing')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 2 — Crop Non-Gunshot Files to Random Gunshot Durations

In [ ]:
def collect_wav_files(directories):
    """Recursively collect all .wav files from a list of directories."""
    files = []
    for d in directories:
        for f in d.rglob('*.wav'):
            files.append(str(f))
    return sorted(files)


def random_crop_audio(filepath, target_samples, sr=SR):
    """
    Load audio and randomly crop to target_samples length.
    If the file is shorter than target, return it as-is.
    """
    y, _ = librosa.load(filepath, sr=sr)
    
    if len(y) <= target_samples:
        # File is already shorter or equal — return as-is
        return y
    
    # Random start point so the crop captures different parts of the audio
    max_start = len(y) - target_samples
    start = random.randint(0, max_start)
    return y[start:start + target_samples]


# Collect non-gunshot files
class0_files = collect_wav_files(CLASS_0_DIRS)
print(f'Found {len(class0_files)} non-gunshot (class 0) files to crop.')

In [ ]:
# Create output directory
TRIMMED_NONGUNSHOTS_DIR.mkdir(parents=True, exist_ok=True)

# Crop each non-gunshot file to a random gunshot duration
trimmed_nongunshot_records = []  # (output_path, duration_ms)
skipped = 0

for filepath in tqdm(class0_files, desc='Cropping non-gunshots'):
    try:
        # Sample a random duration from the gunshot pool
        target_samples = random.choice(gunshot_durations_samples)
        
        # Crop
        cropped = random_crop_audio(filepath, target_samples)
        
        # Build unique output filename
        stem = Path(filepath).stem
        parent = Path(filepath).parent.name
        out_name = f'{parent}__{stem}.wav'
        out_path = TRIMMED_NONGUNSHOTS_DIR / out_name
        
        # Save
        sf.write(str(out_path), cropped, SR)
        dur_ms = round((len(cropped) / SR) * 1000, 2)
        trimmed_nongunshot_records.append((str(out_path), dur_ms))
        
    except Exception as e:
        skipped += 1
        if skipped <= 5:
            print(f'  [SKIP] {Path(filepath).name}: {e}')

print(f'\nCropped {len(trimmed_nongunshot_records)} non-gunshot files.')
print(f'Skipped: {skipped}')
print(f'Output dir: {TRIMMED_NONGUNSHOTS_DIR}')

In [ ]:
# Verify: visualize the new duration distributions
new_nongunshot_durations = [d for _, d in trimmed_nongunshot_records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before
axes[0].hist(gunshot_durations_ms, bins=60, color='#e74c3c', alpha=0.6, label='Gunshot (class 1)', edgecolor='black')
axes[0].axvline(5000, color='#3498db', linewidth=3, linestyle='--', label='Non-gunshot: ALL at 5000ms')
axes[0].set_title('BEFORE Balancing')
axes[0].set_xlabel('Duration (ms)')
axes[0].set_ylabel('Count')
axes[0].legend()

# After
axes[1].hist(gunshot_durations_ms, bins=60, color='#e74c3c', alpha=0.6, label='Gunshot (class 1)', edgecolor='black')
axes[1].hist(new_nongunshot_durations, bins=60, color='#3498db', alpha=0.5, label='Non-gunshot (class 0) — cropped', edgecolor='black')
axes[1].set_title('AFTER Balancing')
axes[1].set_xlabel('Duration (ms)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Non-gunshot duration stats (ms) — AFTER cropping:')
print(f'  Min    : {min(new_nongunshot_durations):.0f} ms')
print(f'  Max    : {max(new_nongunshot_durations):.0f} ms')
print(f'  Mean   : {np.mean(new_nongunshot_durations):.0f} ms')
print(f'  Median : {np.median(new_nongunshot_durations):.0f} ms')

## Step 3 — Re-Extract MFCC Features

In [ ]:
def extract_mfcc(filepath):
    """Extract MFCC features — MUST match audio_pipeline.py exactly."""
    try:
        y, sr = librosa.load(filepath, sr=SR)
        
        # Pad or truncate to MAX_LEN samples (4 seconds)
        if len(y) < MAX_LEN:
            y = np.pad(y, (0, MAX_LEN - len(y)))
        else:
            y = y[:MAX_LEN]
        
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        return np.concatenate([mfcc_mean, mfcc_std])
    except Exception as e:
        print(f'  [WARN] {filepath}: {e}')
        return None


def compute_duration_ms(filepath):
    """Get duration of a wav file in milliseconds."""
    try:
        y, sr = librosa.load(filepath, sr=SR)
        return round((len(y) / sr) * 1000, 2)
    except Exception:
        return 0.0


print('Feature extraction functions loaded.')

In [ ]:
records = []
n_features = N_MFCC * 2  # 80 features (40 mean + 40 std)

# ── Class 0: Trimmed non-gunshots ──
print('Extracting features from trimmed non-gunshot files...')
for fpath, dur_ms in tqdm(trimmed_nongunshot_records, desc='Class 0 MFCCs'):
    feat = extract_mfcc(fpath)
    if feat is not None:
        records.append({
            'file_path': fpath,
            'trimmed_duration_ms': dur_ms,
            'label': 0,
            **{f'mfcc_{i}': feat[i] for i in range(n_features)},
        })

print(f'  Class 0 records: {len(records)}')

# ── Class 1: Already-trimmed gunshots ──
print('\nExtracting features from trimmed gunshot files...')
count_before = len(records)
for fpath in tqdm(gunshot_files, desc='Class 1 MFCCs'):
    feat = extract_mfcc(str(fpath))
    if feat is not None:
        dur = compute_duration_ms(str(fpath))
        records.append({
            'file_path': str(fpath),
            'trimmed_duration_ms': dur,
            'label': 1,
            **{f'mfcc_{i}': feat[i] for i in range(n_features)},
        })

print(f'  Class 1 records: {len(records) - count_before}')
print(f'  Total records: {len(records)}')

## Step 4 — Generate Balanced CSV

In [ ]:
# Create DataFrame, shuffle, and save
df = pd.DataFrame(records)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Save
df.to_csv(CSV_PATH, index=False)

print(f'\n Balanced dataset saved to: {CSV_PATH}')
print(f'   Total samples: {len(df)}')
print(f'   Class 0 (non-gunshot): {(df["label"] == 0).sum()}')
print(f'   Class 1 (gunshot):     {(df["label"] == 1).sum()}')
print(f'   Features: {n_features} MFCC features')
print(f'   Columns: {list(df.columns[:5])} ... {list(df.columns[-3:])}')

In [ ]:
# Final verification: duration distributions in the saved CSV
fig, ax = plt.subplots(figsize=(10, 5))

class0_durations = df[df['label'] == 0]['trimmed_duration_ms']
class1_durations = df[df['label'] == 1]['trimmed_duration_ms']

ax.hist(class0_durations, bins=60, alpha=0.6, color='#3498db', label=f'Class 0 — Non-gunshot (n={len(class0_durations)})', edgecolor='black')
ax.hist(class1_durations, bins=60, alpha=0.6, color='#e74c3c', label=f'Class 1 — Gunshot (n={len(class1_durations)})', edgecolor='black')
ax.set_xlabel('Duration (ms)')
ax.set_ylabel('Count')
ax.set_title('Duration Distribution in Balanced CSV — Bias REMOVED ')
ax.legend()
plt.tight_layout()
plt.show()

print('\nDuration overlap verification:')
print(f'  Class 0 range: {class0_durations.min():.0f}ms – {class0_durations.max():.0f}ms')
print(f'  Class 1 range: {class1_durations.min():.0f}ms – {class1_durations.max():.0f}ms')
print(f'\n  If these ranges overlap, duration bias is eliminated! ')

In [ ]:
# Preview the DataFrame
df.head(10)